In [22]:
import numpy as np
import pandas as pd
from cobra.io import read_sbml_model
from cobra.util.array import create_stoichiometric_matrix
import efmtool_link.efmtool_extern as efmtool_extern

In [23]:
# Read model

#M model
#model_name = 'M_model'

#PQS model
model_name = 'PQS_model'


In [24]:
model = read_sbml_model('../models/' + model_name + ".xml")

In [25]:
# Create stoichiometric matrix
S = create_stoichiometric_matrix(model)

In [26]:
# Transpose S matrix
D_S = S.transpose()

In [27]:
# Dual identity matrix (v's) and concatenate to D_S
I = np.identity(len(D_S))
D_S = np.concatenate([D_S, I], axis=1)

In [28]:
# Make irreversible matrix from irreversible reactions (z's)
irreversibility = [not rxn.reversibility for rxn in model.reactions]
irreversibility_mat = np.diag(irreversibility).astype(int) * -1
irreversibility_mat = irreversibility_mat[:, irreversibility]

In [29]:
# Concatenate D_S with z's
D_S = np.concatenate([D_S, irreversibility_mat], axis=1)

In [30]:
# Define target reaction and get index

#PQS model target
target_rxn = 'R04'

#M model target
#target_rxn = 'r5'

In [31]:
target = model.reactions.index(target_rxn)

In [32]:
# Secrete target metabolites (w)
# model.reactions.index(target)
w = np.zeros(len(D_S), dtype="int")
w[target] = -1

In [33]:
# Concatenate D_S with w
D_S = np.column_stack((D_S, w))
np.shape(D_S)

(11, 28)

In [34]:
# Decide for each reaction if it is reversible (list of indices, not binary)
rev_u = list(np.ones(len(model.metabolites), dtype="int"))
rev_v = list(np.ones(len(model.reactions), dtype="int"))
rev_z = list(np.zeros(np.shape(irreversibility_mat), dtype="int")[1])
rev_w = [0]
D_rev_rxns = rev_u + rev_v + rev_z + rev_w

In [35]:
# Reaction names
rxn_u = [metabolite.id for metabolite in model.metabolites]
rxn_v = [f"v{x+1}" for x in range(0, len(model.reactions))]
rxn_z = [f"z{x+1}" for x in range(0, len(model.reactions)) if irreversibility[x] == 1]
reaction_names = rxn_u + rxn_v + rxn_z + ["w"]

In [36]:
# Give metabolite IDs
metab_names = [reaction.id for reaction in model.reactions]

In [37]:
print(D_S.shape)
print(len(D_rev_rxns))
print(len(reaction_names))
print(len(metab_names))

(11, 28)
28
28
11


In [38]:
efms = efmtool_extern.calculate_flux_modes(D_S, D_rev_rxns)

2026-05-15  14:56:06.726  main                     INFO     | =====================================================
2026-05-15  14:56:06.726  main                     INFO     | efmtool version 4.7.1, 2009-12-04 18:30:05
2026-05-15  14:56:06.726  main                     INFO     | Copyright (c) 2009, Marco Terzer, Zurich, Switzerland
2026-05-15  14:56:06.729  main                     INFO     | This is free software, !!! NO WARRANTY !!!
2026-05-15  14:56:06.729  main                     INFO     | See LICENCE.txt for redistribution conditions
2026-05-15  14:56:06.729  main                     INFO     | =====================================================
2026-05-15  14:56:06.824  main    efm.impl         INFO     | Elemetary flux mode computation
2026-05-15  14:56:06.824  main    efm.impl         INFO     | Implementation:
2026-05-15  14:56:06.824  main    efm.impl         INFO     | ..algorithm name   : SequentialDoubleDescriptionImpl
2026-05-15  14:56:06.824  main    efm.impl     

In [39]:
print(f"Found {efms.shape[1]} EFMs.")

Found 1062 EFMs.


In [40]:
efm_df = pd.DataFrame(efms, index=reaction_names)
# convert to native endian
efm_df = efm_df.astype(np.float64)
efm_df_w = efm_df.T[efm_df.T['w']!=0]
efm_df_w_min = efm_df_w.copy()
efm_df_w_min = efm_df_w_min.iloc[:, len(model.metabolites):len(model.metabolites)+len(model.reactions)]
efm_df_w_min

,v1,v2,v3,v4,v5,v6,v7,v8,v9,v10,v11
2,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
3,1.0,-1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
7,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,0.0,1.0
9,1.0,-1.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.0,0.0,1.0
13,1.0,-1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1053,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1055,1.0,0.0,0.0,0.0,0.0,-1.0,0.0,-1.0,0.0,1.0,0.0
1056,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,1.0,0.0
1060,1.0,0.0,-1.0,0.0,0.0,0.0,0.0,-1.0,0.0,1.0,0.0


In [41]:
efm_df_w_min.sort_index(axis=1).reset_index()

# Step 1: Set all negative values from irreversible reactions to zero
irrev = [not rxn.reversibility for rxn in model.reactions]
mask = np.array(irrev, dtype=bool)

efm_df_w_min.loc[:, mask] = efm_df_w_min.loc[:, mask].clip(lower=0)

# Step 2: For each row, get column names of nonzero (negative) values
nonzero_cols = [
    frozenset(efm_df_w_min.columns[row != 0])
    for _, row in efm_df_w_min.iterrows()
]

# Remove empty sets (rows that were all zero/positive)
nonzero_cols = [s for s in nonzero_cols if s]

# Step 3: Remove duplicates and supersets
# Keep a set only if no other set is a proper subset of it
unique_sets = list(set(nonzero_cols))  # remove exact duplicates first

minimal_sets = []
for s in unique_sets:
    # Keep s only if there is no other set that is a strict subset of s
    if not any(other < s for other in unique_sets if other != s):
        minimal_sets.append(s)

# Optional: convert back to sorted lists for readability
minimal_sets = sorted([sorted(s) for s in minimal_sets], key=len)

In [42]:
minimal_sets

[['v1'],
 ['v4'],
 ['v10', 'v11'],
 ['v10', 'v5'],
 ['v3', 'v5'],
 ['v5', 'v6'],
 ['v11', 'v6', 'v7'],
 ['v11', 'v3', 'v7']]